# Notebook 04 — Experiments and Selection (Semana 4)

## Objetivo
Executar um conjunto pequeno e rastreável de experimentos comparáveis para **selecionar** a abordagem final que será treinada no Notebook 05 (treino final).  
Este notebook **não** faz treino SOTA; faz sanity + comparação rápida.

## Entradas (do pipeline)
- `data/processed/train.csv`, `val.csv`, `test.csv`
- Contrato do target:
  - Preferir `data/processed/target_config_effective.json` (se existir)
  - Fallback: `data/processed/target_config.json`
- `data/processed/label_map.json` (ordem/nomes de classes)

## Saídas obrigatórias (em `reports/`)
- `reports/experiments_table.csv` (1 linha por experimento, métricas + tempo)
- `reports/experiments_configs.json` (configs completas)
- `reports/selection_decision.md` (decisão explícita e justificada)
- (opcional) `reports/plots/*.png` (confusion matrices)

## Regras
- Tudo relativo ao `PROJECT_ROOT`
- Reprodutível: seed fixa + configs salvas
- Não inventar colunas: usar `label_map.json` + `target_config_effective.json/target_config.json`
- Experimentos rápidos, CPU/GPU com fallback


In [1]:
# CÉLULA 01 — Imports + detecção de torch/torchvision (sem pegadinha)
import os
import json
import time
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from PIL import Image

import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

TORCH_AVAILABLE = False
TORCHVISION_AVAILABLE = False
TORCH_IMPORT_ERROR = None
TORCHVISION_IMPORT_ERROR = None

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    TORCH_AVAILABLE = True
except Exception as e:
    TORCH_IMPORT_ERROR = repr(e)

try:
    from torchvision import transforms
    from torchvision.models import resnet18, ResNet18_Weights
    TORCHVISION_AVAILABLE = True
except Exception as e:
    TORCHVISION_IMPORT_ERROR = repr(e)

print("TORCH_AVAILABLE:", TORCH_AVAILABLE)
if not TORCH_AVAILABLE:
    print("Torch import error:", TORCH_IMPORT_ERROR)

print("TORCHVISION_AVAILABLE:", TORCHVISION_AVAILABLE)
if not TORCHVISION_AVAILABLE:
    print("Torchvision import error:", TORCHVISION_IMPORT_ERROR)


TORCH_AVAILABLE: True
TORCHVISION_AVAILABLE: False
Torchvision import error: ModuleNotFoundError("No module named 'torchvision'")


In [2]:
# CÉLULA 02 — Guard claro: este NB04 roda sklearn-only quando torchvision não existe
USE_TORCH = bool(TORCH_AVAILABLE)
USE_TORCHVISION = bool(TORCHVISION_AVAILABLE)

print("USE_TORCH:", USE_TORCH)
print("USE_TORCHVISION:", USE_TORCHVISION)

if not USE_TORCHVISION:
    print("[INFO] torchvision ausente → Notebook 04 rodando apenas experimentos sklearn (ok para Semana 4).")


USE_TORCH: True
USE_TORCHVISION: False
[INFO] torchvision ausente → Notebook 04 rodando apenas experimentos sklearn (ok para Semana 4).


In [3]:
# CÉLULA 03 — PROJECT_ROOT robusto (padrão NB03) + paths do projeto

def _looks_like_repo_root(p: Path) -> bool:
    return (
        (p / "pimple" / "data" / "raw" / "lesions" / "images").exists()
        and (p / "pimple" / "data" / "processed").exists()
        and (p / "pimple" / "reports").exists()
    )

def find_project_root_robust() -> Path:
    env_root = os.environ.get("PIMPLE_PROJECT_ROOT") or os.environ.get("PROJECT_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if _looks_like_repo_root(p):
            return p
        raise FileNotFoundError(
            f"[ERRO] Env PROJECT_ROOT/PIMPLE_PROJECT_ROOT aponta para {p}, "
            "mas não parece ser a raiz do repo."
        )

    start = Path.cwd().resolve()
    for base in [start, *start.parents]:
        if _looks_like_repo_root(base):
            return base

    raise FileNotFoundError(
        "Não consegui localizar PROJECT_ROOT.\n"
        "Dica: defina os.environ['PIMPLE_PROJECT_ROOT'] = r'CAMINHO_PARA_O_REPO' e rode de novo."
    )

PROJECT_ROOT = find_project_root_robust()
PIMPLE_DIR = PROJECT_ROOT / "pimple"

RAW_DIR = PIMPLE_DIR / "data" / "raw" / "lesions"
IMAGES_DIR = RAW_DIR / "images"
MASKS_DIR = RAW_DIR / "masks"
GT_CSV = RAW_DIR / "GroundTruth.csv"

PROCESSED_DIR = PIMPLE_DIR / "data" / "processed"
REPORTS_DIR = PIMPLE_DIR / "reports"
PLOTS_DIR = REPORTS_DIR / "plots"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("IMAGES_DIR:", IMAGES_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("REPORTS_DIR:", REPORTS_DIR)

assert IMAGES_DIR.exists(), f"IMAGES_DIR não existe: {IMAGES_DIR}"
assert PROCESSED_DIR.exists(), f"PROCESSED_DIR não existe: {PROCESSED_DIR}"


PROJECT_ROOT: C:\Users\win\Documents\GitHub
IMAGES_DIR: C:\Users\win\Documents\GitHub\pimple\data\raw\lesions\images
PROCESSED_DIR: C:\Users\win\Documents\GitHub\pimple\data\processed
REPORTS_DIR: C:\Users\win\Documents\GitHub\pimple\reports


In [4]:
# CÉLULA 04 — Leitura de contratos (preferir effective) + label_map robusto

def read_json(path: Path) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def deep_get(d: Dict[str, Any], keys: List[str]) -> Optional[Any]:
    for k in keys:
        if k in d and d[k] is not None:
            return d[k]
    common_parents = ["target", "target_definition", "target_config", "columns", "schema", "contract", "data"]
    for parent in common_parents:
        if parent in d and isinstance(d[parent], dict):
            for k in keys:
                if k in d[parent] and d[parent][k] is not None:
                    return d[parent][k]
    return None

def normalize_label_map_robust(lm: Any, label_cols: List[str]) -> Tuple[List[str], Dict[str, int], str]:
    if isinstance(lm, dict):
        for k in ["idx_to_label", "index_to_label", "id2label", "classes", "labels"]:
            v = lm.get(k, None)
            if isinstance(v, list) and all(isinstance(x, str) for x in v):
                idx_to_label = v
                return idx_to_label, {lab: i for i, lab in enumerate(idx_to_label)}, f"wrapped_list:{k}"
        for k in ["label_to_idx", "label2id", "label2idx"]:
            v = lm.get(k, None)
            if isinstance(v, dict) and all(isinstance(val, int) for val in v.values()):
                label_to_idx = dict(v)
                idx_to_label = [None] * (max(label_to_idx.values()) + 1)
                for lab, i in label_to_idx.items():
                    idx_to_label[i] = lab
                return idx_to_label, label_to_idx, f"wrapped_dict:{k}"

    if isinstance(lm, dict) and len(lm) > 0 and all(str(k).isdigit() for k in lm.keys()):
        idx_to_label = [lm[str(i)] for i in range(len(lm))]
        return idx_to_label, {lab: i for i, lab in enumerate(idx_to_label)}, "digit_key_dict"

    if isinstance(lm, dict) and len(lm) > 0 and all(isinstance(v, int) for v in lm.values()):
        label_to_idx = dict(lm)
        idx_to_label = [None] * (max(label_to_idx.values()) + 1)
        for lab, i in label_to_idx.items():
            idx_to_label[i] = lab
        return idx_to_label, label_to_idx, "label_to_idx_dict"

    if isinstance(lm, list) and all(isinstance(x, str) for x in lm):
        idx_to_label = lm
        return idx_to_label, {lab: i for i, lab in enumerate(idx_to_label)}, "list"

    if isinstance(lm, dict) and len(lm) > 0 and all(isinstance(v, str) for v in lm.values()):
        keys = set(lm.keys())
        if all(c in keys for c in label_cols):
            idx_to_label = [lm[c] for c in label_cols]
            return idx_to_label, {lab: i for i, lab in enumerate(idx_to_label)}, "col_to_label_aligned_by_label_cols"
        lower_map = {str(k).lower(): v for k, v in lm.items()}
        if all(str(c).lower() in lower_map for c in label_cols):
            idx_to_label = [lower_map[str(c).lower()] for c in label_cols]
            return idx_to_label, {lab: i for i, lab in enumerate(idx_to_label)}, "col_to_label_lower_aligned_by_label_cols"

    idx_to_label = list(label_cols)
    return idx_to_label, {lab: i for i, lab in enumerate(idx_to_label)}, "fallback_use_label_cols"

target_cfg_effective_path = PROCESSED_DIR / "target_config_effective.json"
target_cfg_path = PROCESSED_DIR / "target_config.json"
label_map_path = PROCESSED_DIR / "label_map.json"

assert label_map_path.exists(), f"label_map.json não encontrado: {label_map_path}"
label_map = read_json(label_map_path)

if target_cfg_effective_path.exists():
    target_cfg = read_json(target_cfg_effective_path)
    target_cfg_source = str(target_cfg_effective_path.relative_to(PROJECT_ROOT))
else:
    assert target_cfg_path.exists(), f"target_config.json não encontrado: {target_cfg_path}"
    target_cfg = read_json(target_cfg_path)
    target_cfg_source = str(target_cfg_path.relative_to(PROJECT_ROOT))

print("Target config source:", target_cfg_source)

mode = deep_get(target_cfg, ["mode", "task_mode"])
target_encoding = deep_get(target_cfg, ["target_encoding", "encoding", "label_encoding"])
label_cols = deep_get(target_cfg, ["label_cols", "labels", "label_columns", "target_cols"])
image_col = deep_get(target_cfg, ["image_col", "image_column", "image", "filename_col", "path_col"])

if isinstance(label_cols, tuple):
    label_cols = list(label_cols)

assert isinstance(label_cols, list) and len(label_cols) > 1, f"label_cols inválido: {label_cols}"
assert isinstance(image_col, str) and len(image_col) > 0, f"image_col inválido: {image_col}"

if mode is None:
    print("[WARN] mode não encontrado. Assumindo single_label (coerente com one-hot).")
else:
    assert mode == "single_label", f"Notebook 04 assume single_label. Encontrado: {mode}"

if target_encoding is None:
    print("[WARN] target_encoding não encontrado. Prosseguindo.")
else:
    if target_encoding != "one_hot_multiclass":
        print("[WARN] target_encoding diferente do esperado:", target_encoding)

IDX_TO_LABEL, LABEL_TO_IDX, LABEL_MAP_STRATEGY = normalize_label_map_robust(label_map, label_cols)

print("mode:", mode)
print("target_encoding:", target_encoding)
print("image_col:", image_col)
print("label_cols:", label_cols)
print("label_map strategy:", LABEL_MAP_STRATEGY)
print("labels:", IDX_TO_LABEL)

mask_policy = None
if isinstance(target_cfg, dict):
    mask_policy = target_cfg.get("mask_policy", None)
    if mask_policy is None and isinstance(target_cfg.get("mask"), dict):
        mask_policy = target_cfg["mask"].get("policy", None)
print("mask_policy:", mask_policy)


Target config source: pimple\data\processed\target_config_effective.json
[WARN] target_encoding não encontrado. Prosseguindo.
mode: single_label
target_encoding: None
image_col: image_stem
label_cols: ['mel', 'nv', 'bcc', 'akiec', 'bkl', 'df', 'vasc']
label_map strategy: wrapped_list:index_to_label
labels: ['mel', 'nv', 'bcc', 'akiec', 'bkl', 'df', 'vasc']
mask_policy: {'mask_coverage_ratio_in_clean': 1.0, 'mask_required_for_inclusion': False, 'notes': 'Máscaras são opcionais no dataset_clean; usadas apenas se a trilha de segmentação for escolhida.', 'resolution_strategy': {'fallback_by_stem': True, 'from_csv_if_available': False, 'suffixes_used': ['', '_mask', '-mask', '_seg', '-seg', '_segmentation', '-segmentation', '_lesion', '_lesion_mask', '_binary', '_annotation', '_ann']}, 'segmentation_considered': True}


In [5]:
# CÉLULA 05 — Carregar splits + one-hot -> class index y
train_path = PROCESSED_DIR / "train.csv"
val_path = PROCESSED_DIR / "val.csv"
test_path = PROCESSED_DIR / "test.csv"

assert train_path.exists() and val_path.exists() and test_path.exists(), "train/val/test.csv não encontrados em data/processed/"

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    assert image_col in df.columns, f"[{name}] falta image_col={image_col}"
    missing = [c for c in label_cols if c not in df.columns]
    assert len(missing) == 0, f"[{name}] faltam label_cols: {missing}"

print("Shapes:", train_df.shape, val_df.shape, test_df.shape)

def one_hot_to_y(df: pd.DataFrame, cols: List[str]) -> np.ndarray:
    y = df[cols].values
    s = y.sum(axis=1)
    if not np.allclose(s, 1.0, atol=1e-6):
        bad = np.where(~np.isclose(s, 1.0, atol=1e-6))[0][:10]
        raise ValueError(f"one-hot inválido: soma != 1. Ex idx: {bad}")
    return np.argmax(y, axis=1).astype(int)

y_train = one_hot_to_y(train_df, label_cols)
y_val = one_hot_to_y(val_df, label_cols)
y_test = one_hot_to_y(test_df, label_cols)

print("y_train distribution:", np.bincount(y_train, minlength=len(label_cols)))


Shapes: (7011, 12) (1502, 12) (1502, 12)
y_train distribution: [ 779 4693  360  229  769   81  100]


In [6]:
# CÉLULA 06 — Seed fixa + device + Timer
GLOBAL_SEED = int(target_cfg.get("seed", 42))

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    if TORCH_AVAILABLE:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(GLOBAL_SEED)

DEVICE = "cpu"
if TORCH_AVAILABLE and torch.cuda.is_available():
    DEVICE = "cuda"
elif TORCH_AVAILABLE and getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = "mps"

print("GLOBAL_SEED:", GLOBAL_SEED)
print("DEVICE:", DEVICE)

@dataclass
class Timer:
    start: float = 0.0
    end: float = 0.0
    def __enter__(self):
        self.start = time.time()
        return self
    def __exit__(self, exc_type, exc, tb):
        self.end = time.time()
    @property
    def seconds(self) -> float:
        return float(self.end - self.start)


GLOBAL_SEED: 42
DEVICE: cpu


In [7]:
# CÉLULA 07 — Métricas + plot confusion matrix
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, n_classes: int) -> Dict[str, Any]:
    acc = float(accuracy_score(y_true, y_pred))
    f1m = float(f1_score(y_true, y_pred, average="macro"))
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n_classes)))
    return {"accuracy": acc, "f1_macro": f1m, "confusion_matrix": cm}

def plot_confusion_matrix(cm: np.ndarray, labels: List[str], title: str, save_path: Path) -> None:
    fig = plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(labels))
    plt.xticks(tick_marks, labels, rotation=45, ha="right")
    plt.yticks(tick_marks, labels)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    fig.savefig(save_path, dpi=160, bbox_inches="tight")
    plt.close(fig)


In [8]:
# CÉLULA 08 — resolve_image_path robusto (padrão NB03)
ALLOWED_IMAGE_EXTS = [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"]

def resolve_image_path(img_value: Any) -> Path:
    if pd.isna(img_value):
        raise ValueError("image_col contém NaN.")

    s = str(img_value).strip().replace("\\", "/")
    p = Path(s)

    # 0) se já existe (absoluto/relativo ao cwd), usa direto
    try:
        p2 = p.expanduser()
        if p2.exists():
            return p2.resolve()
    except Exception:
        pass

    # 1) basename
    fname = p.name
    c1 = IMAGES_DIR / fname
    if c1.exists():
        return c1

    # 2) relativo dentro de images/
    c2 = IMAGES_DIR / s
    if c2.exists():
        return c2

    # 3) stem
    stem = Path(fname).stem
    for ext in ALLOWED_IMAGE_EXTS:
        c3 = IMAGES_DIR / f"{stem}{ext}"
        if c3.exists():
            return c3

    # 4) glob fallback
    matches = sorted(IMAGES_DIR.glob(f"{stem}.*"))
    if matches:
        return matches[0]

    raise FileNotFoundError(f"Imagem não encontrada para valor={img_value}. Tentativas: {c1}, {c2}, stem={stem}")

# sanity
ok = 0
for v in train_df[image_col].head(20).tolist():
    try:
        _ = resolve_image_path(v)
        ok += 1
    except Exception as e:
        print("FAIL sample:", v, "=>", repr(e))
print(f"[SANITY] resolve_image_path: {ok}/20 resolvidas.")


[SANITY] resolve_image_path: 20/20 resolvidas.


In [9]:
# CÉLULA 09 — Features para sklearn (flatten)
def load_image_as_feature(img_path: Path, size: int) -> np.ndarray:
    img = Image.open(img_path).convert("RGB").resize((size, size))
    arr = np.asarray(img, dtype=np.float32) / 255.0
    return arr.reshape(-1)

def make_features(df: pd.DataFrame, size: int) -> np.ndarray:
    X = np.zeros((len(df), size * size * 3), dtype=np.float32)
    for i, v in enumerate(df[image_col].values):
        p = resolve_image_path(v)
        X[i] = load_image_as_feature(p, size=size)
    return X


In [10]:
# CÉLULA 10 — fit_eval_sklearn (compatível com versões sem multi_class)
def fit_eval_sklearn(train_df, y_train, val_df, y_val, cfg: Dict[str, Any]) -> Dict[str, Any]:
    size = int(cfg["input_size"])
    class_weight = cfg.get("class_weight", None)  # None ou "balanced"

    Xtr = make_features(train_df, size=size)
    Xva = make_features(val_df, size=size)

    base_params = LogisticRegression().get_params()
    lr_kwargs = {
        "max_iter": int(cfg.get("max_iter", 200)),
        "solver": "lbfgs",
        "class_weight": class_weight,
        "random_state": int(cfg.get("seed", GLOBAL_SEED)),
    }
    lr_kwargs = {k: v for k, v in lr_kwargs.items() if k in base_params}

    pipe = Pipeline([
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("clf", LogisticRegression(**lr_kwargs)),
    ])

    pipe.fit(Xtr, y_train)
    y_pred = pipe.predict(Xva)

    m = compute_metrics(y_val, y_pred, n_classes=len(label_cols))
    return {
        "model_obj": pipe,
        "val_metrics": {"accuracy": m["accuracy"], "f1_macro": m["f1_macro"]},
        "val_confusion_matrix": m["confusion_matrix"],
    }


In [11]:
# CÉLULA 11 — Experimentos (sklearn-only; torch_resnet18 removido para evitar armadilhas)
EXPERIMENTS: Dict[str, Dict[str, Any]] = {
    "sk_logreg_32": {
        "family": "sklearn_logreg_flatten",
        "input_size": 32,
        "class_weight": None,
        "max_iter": 200,
        "seed": GLOBAL_SEED,
    },
    "sk_logreg_32_bal": {
        "family": "sklearn_logreg_flatten",
        "input_size": 32,
        "class_weight": "balanced",
        "max_iter": 200,
        "seed": GLOBAL_SEED,
    },
    "sk_logreg_64": {
        "family": "sklearn_logreg_flatten",
        "input_size": 64,
        "class_weight": None,
        "max_iter": 200,
        "seed": GLOBAL_SEED,
    },
    "sk_logreg_64_bal": {
        "family": "sklearn_logreg_flatten",
        "input_size": 64,
        "class_weight": "balanced",
        "max_iter": 200,
        "seed": GLOBAL_SEED,
    },
}

print("Experiments:", list(EXPERIMENTS.keys()))


Experiments: ['sk_logreg_32', 'sk_logreg_32_bal', 'sk_logreg_64', 'sk_logreg_64_bal']


In [12]:
# CÉLULA 12 — Runner único (somente sklearn) + registro de CM e runtime
def run_one_experiment(exp_id: str, cfg: Dict[str, Any]) -> Dict[str, Any]:
    cfg_full = dict(cfg)
    cfg_full["exp_id"] = exp_id
    cfg_full["target_config_source"] = target_cfg_source
    cfg_full["project_root"] = str(PROJECT_ROOT)
    cfg_full["device"] = DEVICE
    cfg_full["labels"] = IDX_TO_LABEL
    cfg_full["mask_policy"] = mask_policy

    family = cfg_full.get("family")

    with Timer() as t:
        if family == "sklearn_logreg_flatten":
            out = fit_eval_sklearn(train_df, y_train, val_df, y_val, cfg_full)
        else:
            raise ValueError(f"Família desconhecida: {family}")

    rec = {
        "experiment_id": exp_id,
        "family": family,
        "input_size": int(cfg_full.get("input_size")),
        "normalization": cfg_full.get("normalization", None),
        "class_weight": cfg_full.get("class_weight", None),
        "seed": int(cfg_full.get("seed", GLOBAL_SEED)),
        "device": DEVICE,
        "val_accuracy": float(out["val_metrics"]["accuracy"]),
        "val_f1_macro": float(out["val_metrics"]["f1_macro"]),
        "runtime_sec": float(t.seconds),
        "val_confusion_matrix_path": None,
    }

    cm = out.get("val_confusion_matrix")
    if cm is not None:
        cm_path = PLOTS_DIR / f"cm_{exp_id}.png"
        plot_confusion_matrix(cm, IDX_TO_LABEL, title=f"Confusion Matrix (val) — {exp_id}", save_path=cm_path)
        rec["val_confusion_matrix_path"] = str(cm_path.relative_to(PROJECT_ROOT))

    return {"record": rec, "config": cfg_full}


In [13]:
# CÉLULA 13 — Executar todos os experimentos (uma única execução) + ordenar por F1 macro
results: List[Dict[str, Any]] = []
configs_out: Dict[str, Any] = {}

for exp_id, cfg in EXPERIMENTS.items():
    print("\n==============================")
    print("RUN:", exp_id)
    print("==============================")
    try:
        r = run_one_experiment(exp_id, cfg)
        results.append(r["record"])
        configs_out[exp_id] = r["config"]
        print("OK:", exp_id, "| val_f1_macro=", r["record"]["val_f1_macro"])
    except Exception as e:
        results.append({
            "experiment_id": exp_id,
            "family": cfg.get("family"),
            "input_size": cfg.get("input_size"),
            "normalization": cfg.get("normalization", None),
            "class_weight": cfg.get("class_weight", None),
            "seed": int(cfg.get("seed", GLOBAL_SEED)),
            "device": DEVICE,
            "val_accuracy": None,
            "val_f1_macro": None,
            "runtime_sec": None,
            "val_confusion_matrix_path": None,
            "error": repr(e),
        })
        configs_out[exp_id] = dict(cfg, exp_id=exp_id, error=repr(e))
        print("FAILED:", exp_id, "|", repr(e))

experiments_df = pd.DataFrame(results)

experiments_df = experiments_df.sort_values(
    by=["val_f1_macro", "val_accuracy", "runtime_sec"],
    ascending=[False, False, True],
    na_position="last"
).reset_index(drop=True)

experiments_df["rank_f1"] = experiments_df["val_f1_macro"].rank(ascending=False, method="min")

experiments_df



RUN: sk_logreg_32


c:\Users\win\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 200 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=200).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


OK: sk_logreg_32 | val_f1_macro= 0.38765920023818506

RUN: sk_logreg_32_bal


c:\Users\win\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 200 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=200).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


OK: sk_logreg_32_bal | val_f1_macro= 0.35949142732075845

RUN: sk_logreg_64


c:\Users\win\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 200 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=200).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


OK: sk_logreg_64 | val_f1_macro= 0.34354410957322873

RUN: sk_logreg_64_bal


c:\Users\win\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 200 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=200).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


OK: sk_logreg_64_bal | val_f1_macro= 0.3618675275816507


,experiment_id,family,input_size,normalization,class_weight,seed,device,val_accuracy,val_f1_macro,runtime_sec,val_confusion_matrix_path,rank_f1
0,sk_logreg_32,sklearn_logreg_flatten,32,None,None,42,cpu,0.657124,0.387659,32.107437,pimple\reports\plots\cm_sk_logreg_32.png,1.0
1,sk_logreg_64_bal,sklearn_logreg_flatten,64,None,balanced,42,cpu,0.609188,0.361868,38.929318,pimple\reports\plots\cm_sk_logreg_64_bal.png,2.0
2,sk_logreg_32_bal,sklearn_logreg_flatten,32,None,balanced,42,cpu,0.571238,0.359491,31.460962,pimple\reports\plots\cm_sk_logreg_32_bal.png,3.0
3,sk_logreg_64,sklearn_logreg_flatten,64,None,None,42,cpu,0.631824,0.343544,40.000700,pimple\reports\plots\cm_sk_logreg_64.png,4.0


In [14]:
# CÉLULA 14 — Salvar entregáveis obrigatórios: experiments_table.csv + experiments_configs.json
table_path = REPORTS_DIR / "experiments_table.csv"
configs_path = REPORTS_DIR / "experiments_configs.json"

experiments_df.to_csv(table_path, index=False)

payload = {
    "meta": {
        "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "project_root": str(PROJECT_ROOT),
        "target_config_source": target_cfg_source,
        "label_cols": label_cols,
        "image_col": image_col,
        "labels": IDX_TO_LABEL,
        "seed": GLOBAL_SEED,
        "device": DEVICE,
        "torch_available": TORCH_AVAILABLE,
        "torchvision_available": TORCHVISION_AVAILABLE,
        "mask_policy": mask_policy,
    },
    "experiments": configs_out,
}

with open(configs_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)

print("Saved:", table_path.relative_to(PROJECT_ROOT))
print("Saved:", configs_path.relative_to(PROJECT_ROOT))


Saved: pimple\reports\experiments_table.csv
Saved: pimple\reports\experiments_configs.json


In [15]:
# CÉLULA 15 — Selection Decision com TRILHA explícita (A/B/C) + justificativa segmentação
decision_path = REPORTS_DIR / "selection_decision.md"

valid = experiments_df.dropna(subset=["val_f1_macro"]).copy()

lines = []
lines.append("# Selection Decision — Notebook 04 (Semana 4)\n")
lines.append("## Contexto\n")
lines.append("- Objetivo: selecionar abordagem para treino final no Notebook 05, com experimentos rápidos e comparáveis.\n")
lines.append(f"- Contrato do target: `{target_cfg_source}`\n")
lines.append(f"- Seed global: `{GLOBAL_SEED}`\n")
lines.append(f"- Device: `{DEVICE}`\n")
if mask_policy is not None:
    lines.append(f"- mask_policy (contrato): `{mask_policy}`\n")

lines.append("\n## Critério de seleção\n")
lines.append("- Primário: **F1 macro em validação**\n")
lines.append("- Secundário: accuracy e simplicidade operacional (preprocess + estabilidade + tempo)\n")

lines.append("\n## Trilha selecionada\n")
lines.append("**Trilha A: somente classificação.**\n")
lines.append("- Justificativa: os experimentos desta Semana 4 focaram em classificação (single-label multiclass) para selecionar uma config de treino final.\n")
lines.append("- Segmentação: não foi executada neste Notebook 04 para manter a etapa rápida e comparável; será considerada em etapa própria caso o `mask_policy` e a qualidade das máscaras suportem.\n")

if len(valid) == 0:
    lines.append("\n## Resultado\n")
    lines.append("**Nenhum experimento produziu métricas válidas.**\n\n")
    if "error" in experiments_df.columns:
        lines.append("### Erros observados (resumo)\n")
        vc = experiments_df["error"].value_counts(dropna=True).head(10)
        for k, v in vc.items():
            lines.append(f"- {k}: {v}\n")
    lines.append("\n## Próximos passos\n")
    lines.append("- Corrigir causa raiz e reexecutar o Notebook 04.\n")
else:
    best = valid.iloc[0].to_dict()
    best_id = best["experiment_id"]
    best_cfg = configs_out.get(best_id, {})

    lines.append("\n## Resultado (melhor experimento)\n")
    lines.append(f"- **experiment_id**: `{best_id}`\n")
    lines.append(f"- family: `{best.get('family')}`\n")
    lines.append(f"- input_size: `{best.get('input_size')}`\n")
    lines.append(f"- class_weight: `{best.get('class_weight')}`\n")
    lines.append(f"- val_f1_macro: `{best.get('val_f1_macro')}`\n")
    lines.append(f"- val_accuracy: `{best.get('val_accuracy')}`\n")
    lines.append(f"- runtime_sec: `{best.get('runtime_sec')}`\n")
    if best.get("val_confusion_matrix_path"):
        lines.append(f"- confusion_matrix plot: `{best.get('val_confusion_matrix_path')}`\n")

    lines.append("\n## Config escolhida (para Notebook 05)\n```json\n")
    lines.append(json.dumps(best_cfg, indent=2, ensure_ascii=False))
    lines.append("\n```\n")

    lines.append("\n## Justificativa\n")
    lines.append("- A escolha prioriza F1 macro (robusto a desbalanceamento) e mantém pipeline simples e reproduzível.\n")
    lines.append("- Observação: como este NB04 está em modo sklearn-only (torchvision ausente), os resultados são adequados como sanity e seleção preliminar.\n")

    lines.append("\n## Próximos passos (Notebook 05)\n")
    lines.append(f"- Carregar `reports/experiments_configs.json` e usar a config do experimento `{best_id}`.\n")
    lines.append("- Treinar com mais rigor (checkpoint do melhor, avaliação em test, export do pacote em `models/`).\n")

with open(decision_path, "w", encoding="utf-8") as f:
    f.write("".join(lines))

print("Saved:", decision_path.relative_to(PROJECT_ROOT))
if len(valid) > 0:
    print("Selected best experiment:", valid.iloc[0]["experiment_id"])


Saved: pimple\reports\selection_decision.md
Selected best experiment: sk_logreg_32


In [16]:
# CÉLULA FINAL 01 — OVERRIDE DEFINITIVO (remove duplicação na prática)
# Cole esta célula NO FINAL do notebook. Ela sobrescreve qualquer EXPERIMENTS/runner anterior.

from sklearn.svm import LinearSVC

# Experimentos (sklearn-only, rápidos e comparáveis)
# Incluo 1 família adicional (LinearSVC) pra tentar ganhar um pouco de F1 sem torch/torchvision.
EXPERIMENTS = {
    "sk_logreg_32": {
        "family": "sklearn_logreg_flatten",
        "input_size": 32,
        "class_weight": None,
        "max_iter": 250,
        "seed": GLOBAL_SEED,
    },
    "sk_logreg_32_bal": {
        "family": "sklearn_logreg_flatten",
        "input_size": 32,
        "class_weight": "balanced",
        "max_iter": 250,
        "seed": GLOBAL_SEED,
    },
    "sk_logreg_64": {
        "family": "sklearn_logreg_flatten",
        "input_size": 64,
        "class_weight": None,
        "max_iter": 250,
        "seed": GLOBAL_SEED,
    },
    "sk_logreg_64_bal": {
        "family": "sklearn_logreg_flatten",
        "input_size": 64,
        "class_weight": "balanced",
        "max_iter": 250,
        "seed": GLOBAL_SEED,
    },
    # Extra: linear SVM (às vezes melhora macro-F1 em desbalanceamento)
    "svm_64_bal": {
        "family": "sklearn_linearsvc_flatten",
        "input_size": 64,
        "class_weight": "balanced",
        "C": 1.0,
        "seed": GLOBAL_SEED,
    },
}

print("[OVERRIDE] Experiments finais:", list(EXPERIMENTS.keys()))

def fit_eval_linearsvc(train_df, y_train, val_df, y_val, cfg: Dict[str, Any]) -> Dict[str, Any]:
    size = int(cfg["input_size"])
    class_weight = cfg.get("class_weight", None)
    C = float(cfg.get("C", 1.0))

    Xtr = make_features(train_df, size=size)
    Xva = make_features(val_df, size=size)

    # LinearSVC não dá predict_proba, mas funciona bem em alta dimensão
    clf = Pipeline([
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("clf", LinearSVC(class_weight=class_weight, C=C)),
    ])
    clf.fit(Xtr, y_train)
    y_pred = clf.predict(Xva)

    m = compute_metrics(y_val, y_pred, n_classes=len(label_cols))
    return {
        "model_obj": clf,
        "val_metrics": {"accuracy": m["accuracy"], "f1_macro": m["f1_macro"]},
        "val_confusion_matrix": m["confusion_matrix"],
    }

def run_one_experiment(exp_id: str, cfg: Dict[str, Any]) -> Dict[str, Any]:
    cfg_full = dict(cfg)
    cfg_full["exp_id"] = exp_id
    cfg_full["target_config_source"] = target_cfg_source
    cfg_full["project_root"] = str(PROJECT_ROOT)
    cfg_full["device"] = DEVICE
    cfg_full["labels"] = IDX_TO_LABEL
    cfg_full["mask_policy"] = mask_policy

    family = cfg_full.get("family")

    with Timer() as t:
        if family == "sklearn_logreg_flatten":
            out = fit_eval_sklearn(train_df, y_train, val_df, y_val, cfg_full)
        elif family == "sklearn_linearsvc_flatten":
            out = fit_eval_linearsvc(train_df, y_train, val_df, y_val, cfg_full)
        else:
            raise ValueError(f"Família desconhecida: {family}")

    rec = {
        "experiment_id": exp_id,
        "family": family,
        "input_size": int(cfg_full.get("input_size")),
        "normalization": cfg_full.get("normalization", None),
        "class_weight": cfg_full.get("class_weight", None),
        "seed": int(cfg_full.get("seed", GLOBAL_SEED)),
        "device": DEVICE,
        "val_accuracy": float(out["val_metrics"]["accuracy"]),
        "val_f1_macro": float(out["val_metrics"]["f1_macro"]),
        "runtime_sec": float(t.seconds),
        "val_confusion_matrix_path": None,
        "error": None,
    }

    cm = out.get("val_confusion_matrix")
    if cm is not None:
        cm_path = PLOTS_DIR / f"cm_{exp_id}.png"
        plot_confusion_matrix(cm, IDX_TO_LABEL, title=f"Confusion Matrix (val) — {exp_id}", save_path=cm_path)
        rec["val_confusion_matrix_path"] = str(cm_path.relative_to(PROJECT_ROOT))

    return {"record": rec, "config": cfg_full}


[OVERRIDE] Experiments finais: ['sk_logreg_32', 'sk_logreg_32_bal', 'sk_logreg_64', 'sk_logreg_64_bal', 'svm_64_bal']


In [17]:
# CÉLULA FINAL 02 — EXECUÇÃO ÚNICA + SALVAR (artefatos oficiais da Semana 4)
# Esta célula roda UMA vez e salva reports/experiments_table.csv e reports/experiments_configs.json

results = []
configs_out = {}

for exp_id, cfg in EXPERIMENTS.items():
    print("\n==============================")
    print("RUN:", exp_id)
    print("==============================")
    try:
        r = run_one_experiment(exp_id, cfg)
        results.append(r["record"])
        configs_out[exp_id] = r["config"]
        print("OK:", exp_id, "| val_f1_macro=", r["record"]["val_f1_macro"])
    except Exception as e:
        results.append({
            "experiment_id": exp_id,
            "family": cfg.get("family"),
            "input_size": cfg.get("input_size"),
            "normalization": cfg.get("normalization", None),
            "class_weight": cfg.get("class_weight", None),
            "seed": int(cfg.get("seed", GLOBAL_SEED)),
            "device": DEVICE,
            "val_accuracy": None,
            "val_f1_macro": None,
            "runtime_sec": None,
            "val_confusion_matrix_path": None,
            "error": repr(e),
        })
        configs_out[exp_id] = dict(cfg, exp_id=exp_id, error=repr(e))
        print("FAILED:", exp_id, "|", repr(e))

experiments_df = pd.DataFrame(results)

experiments_df = experiments_df.sort_values(
    by=["val_f1_macro", "val_accuracy", "runtime_sec"],
    ascending=[False, False, True],
    na_position="last"
).reset_index(drop=True)

experiments_df["rank_f1"] = experiments_df["val_f1_macro"].rank(ascending=False, method="min")

# salvar tabela e configs
table_path = REPORTS_DIR / "experiments_table.csv"
configs_path = REPORTS_DIR / "experiments_configs.json"

experiments_df.to_csv(table_path, index=False)

payload = {
    "meta": {
        "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "project_root": str(PROJECT_ROOT),
        "target_config_source": target_cfg_source,
        "label_cols": label_cols,
        "image_col": image_col,
        "labels": IDX_TO_LABEL,
        "seed": GLOBAL_SEED,
        "device": DEVICE,
        "torch_available": TORCH_AVAILABLE,
        "torchvision_available": TORCHVISION_AVAILABLE,
        "mask_policy": mask_policy,
        "note": "sklearn-only (torchvision ausente). Resultados são sanity/seleção preliminar.",
    },
    "experiments": configs_out,
}

with open(configs_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)

print("Saved:", table_path.relative_to(PROJECT_ROOT))
print("Saved:", configs_path.relative_to(PROJECT_ROOT))
experiments_df.head(10)



RUN: sk_logreg_32


c:\Users\win\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 250 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=250).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


OK: sk_logreg_32 | val_f1_macro= 0.38160003715417934

RUN: sk_logreg_32_bal


c:\Users\win\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 250 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=250).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


OK: sk_logreg_32_bal | val_f1_macro= 0.36343033310923706

RUN: sk_logreg_64


...

c:\Users\win\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 250 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=250).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


OK: sk_logreg_64_bal | val_f1_macro= 0.3506980386485289

RUN: svm_64_bal


c:\Users\win\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


OK: svm_64_bal | val_f1_macro= 0.29746953409041493
Saved: pimple\reports\experiments_table.csv
Saved: pimple\reports\experiments_configs.json


,experiment_id,family,input_size,normalization,class_weight,seed,device,val_accuracy,val_f1_macro,runtime_sec,val_confusion_matrix_path,error,rank_f1
0,sk_logreg_32,sklearn_logreg_flatten,32,None,None,42,cpu,0.657124,0.381600,30.750239,pimple\reports\plots\cm_sk_logreg_32.png,None,1.0
1,sk_logreg_32_bal,sklearn_logreg_flatten,32,None,balanced,42,cpu,0.577230,0.363430,31.517815,pimple\reports\plots\cm_sk_logreg_32_bal.png,None,2.0
2,sk_logreg_64_bal,sklearn_logreg_flatten,64,None,balanced,42,cpu,0.617177,0.350698,40.966924,pimple\reports\plots\cm_sk_logreg_64_bal.png,None,3.0
3,sk_logreg_64,sklearn_logreg_flatten,64,None,None,42,cpu,0.633822,0.336021,42.925692,pimple\reports\plots\cm_sk_logreg_64.png,None,4.0
4,svm_64_bal,sklearn_linearsvc_flatten,64,None,balanced,42,cpu,0.621838,0.297470,265.270951,pimple\reports\plots\cm_svm_64_bal.png,None,5.0


In [18]:
# CÉLULA FINAL 03 — SELECTION DECISION (TRILHA A/B/C EXPLÍCITA) + justificativa do F1 baixo e do torch ausente

decision_path = REPORTS_DIR / "selection_decision.md"
valid = experiments_df.dropna(subset=["val_f1_macro"]).copy()

lines = []
lines.append("# Selection Decision — Notebook 04 (Semana 4)\n")
lines.append("## Contexto\n")
lines.append("- Objetivo: selecionar abordagem para treino final no Notebook 05, com experimentos rápidos e comparáveis.\n")
lines.append(f"- Contrato do target: `{target_cfg_source}`\n")
lines.append(f"- Seed global: `{GLOBAL_SEED}`\n")
lines.append(f"- Device: `{DEVICE}`\n")
if mask_policy is not None:
    lines.append(f"- mask_policy (contrato): `{mask_policy}`\n")

lines.append("\n## Critério de seleção\n")
lines.append("- Primário: **F1 macro em validação**\n")
lines.append("- Secundário: accuracy e simplicidade operacional (preprocess + estabilidade + tempo)\n")

# ✅ TRILHA explícita (o plano exige)
lines.append("\n## Trilha selecionada\n")
lines.append("**Trilha A: somente classificação.**\n")
lines.append("- Justificativa: nesta Semana 4 os experimentos executados foram de classificação para escolher uma config rastreável para o treino final.\n")
lines.append("- Segmentação: não foi executada no Notebook 04 para manter a etapa rápida e comparável; será considerada em etapa própria se a qualidade das máscaras/política suportar.\n")

# Torch/ResNet: explicar claramente por que não foi rodado
lines.append("\n## Observação sobre Torch/ResNet\n")
if not TORCHVISION_AVAILABLE:
    lines.append("- `torchvision` está ausente neste ambiente; portanto, experimentos com ResNet18 pré-treinada (head-only/fine-tune) **não foram executados**.\n")
    lines.append("- Consequência: a seleção aqui é **preliminar** (sanity), baseada em modelos lineares sobre pixels flatten.\n")
else:
    lines.append("- `torchvision` está disponível; (se você habilitar a família ResNet18, ela pode substituir estes baselines).\n")

if len(valid) == 0:
    lines.append("\n## Resultado\n")
    lines.append("**Nenhum experimento produziu métricas válidas.**\n\n")
    if "error" in experiments_df.columns:
        lines.append("### Erros observados (resumo)\n")
        vc = experiments_df["error"].value_counts(dropna=True).head(10)
        for k, v in vc.items():
            lines.append(f"- {k}: {v}\n")
    lines.append("\n## Próximos passos\n")
    lines.append("- Corrigir a causa raiz e reexecutar o bloco FINAL.\n")
else:
    best = valid.iloc[0].to_dict()
    best_id = best["experiment_id"]
    best_cfg = configs_out.get(best_id, {})

    lines.append("\n## Resultado (melhor experimento)\n")
    lines.append(f"- **experiment_id**: `{best_id}`\n")
    lines.append(f"- family: `{best.get('family')}`\n")
    lines.append(f"- input_size: `{best.get('input_size')}`\n")
    lines.append(f"- class_weight: `{best.get('class_weight')}`\n")
    lines.append(f"- val_f1_macro: `{best.get('val_f1_macro')}`\n")
    lines.append(f"- val_accuracy: `{best.get('val_accuracy')}`\n")
    lines.append(f"- runtime_sec: `{best.get('runtime_sec')}`\n")
    if best.get("val_confusion_matrix_path"):
        lines.append(f"- confusion_matrix plot: `{best.get('val_confusion_matrix_path')}`\n")

    lines.append("\n## Config escolhida (para Notebook 05)\n```json\n")
    lines.append(json.dumps(best_cfg, indent=2, ensure_ascii=False))
    lines.append("\n```\n")

    # F1 baixo: registrar explicitamente como sanity
    lines.append("\n## Interpretação dos resultados\n")
    lines.append("- O macro-F1 observado é **baixo** e deve ser interpretado como **sanity check** do pipeline (flatten em baixa resolução + modelo linear).\n")
    lines.append("- Para uma seleção definitiva, recomenda-se habilitar um backbone pré-treinado (ex.: ResNet18 head-only) assim que `torchvision` estiver disponível.\n")

    lines.append("\n## Próximos passos (Notebook 05)\n")
    lines.append(f"- Carregar `reports/experiments_configs.json` e usar a config do experimento `{best_id}` como baseline.\n")
    lines.append("- Se/Quando `torchvision` estiver disponível: repetir Semana 4 com ResNet18 head-only e selecionar com base nessa família.\n")

with open(decision_path, "w", encoding="utf-8") as f:
    f.write("".join(lines))

print("Saved:", decision_path.relative_to(PROJECT_ROOT))
if len(valid) > 0:
    print("Selected best experiment:", valid.iloc[0]["experiment_id"])


Saved: pimple\reports\selection_decision.md
Selected best experiment: sk_logreg_32
